In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline 
import cv2
import os

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from tensorflow.python.keras.applications.resnet import ResNet50
from tensorflow.python.keras.models import Sequential
from tensorflow.python.keras.layers import Dense
from tensorflow.keras import optimizers
from keras.applications.resnet50 import preprocess_input
from keras.preprocessing.image import ImageDataGenerator
from tensorflow.python.keras.callbacks import EarlyStopping, ModelCheckpoint

In [ ]:
NUM_CLASSES = 10

CHANNELS = 3

OBJECTIVE_FUNCTION = 'categorical_crossentropy'

# Métrica de precisão comum para todas as saídas, mas pode usar métricas diferentes para saídas diferentes
LOSS_METRICS = ['accuracy']

# EARLY_STOP_PATIENCE deve ser < NUM_EPOCHS
NUM_EPOCHS = 40
EARLY_STOP_PATIENCE = 10

# O valor dessas etapas deve ser FACTOR adequado de nº de imagens em pastas de treinamento e teste, respectivamente
# As imagens de treinamento processadas em cada etapa seriam no.-of-train-images / STEPS_PER_EPOCH_TRAINING
STEPS_PER_EPOCH_TRAINING   = 8
STEPS_PER_EPOCH_VALIDATION = 2

# O valor dessas etapas deve ser FACTOR adequado de nº de imagens em pastas de treinamento e teste, respectivamente
# NOTE que estes BATCH * são para lotes de Keras ImageDataGenerator para preencher a entrada de etapa de época
# BATCH_SIZE_TRAINING = 100
# BATCH_SIZE_VALIDATION = 100

In [ ]:
image_size = (32, 32)
batch_size = 25

# Usando o aumento de dados de imagem
# Quando você não tem um grande conjunto de dados de imagens, é uma boa prática introduzir artificialmente a diversidade 
# da amostra aplicando transformações aleatórias, porém realistas, às imagens de treinamento, como inversão horizontal 
# aleatória ou pequenas rotações aleatórias. Isso ajuda a expor o modelo a diferentes aspectos dos dados de treinamento, 
# ao mesmo tempo que desacelera o overfitting.

data_generator = ImageDataGenerator(preprocessing_function=preprocess_input)

train_dataset = data_generator.flow_from_directory(
    'cifar-10-data-set-grey-scale-2/train',
    target_size=(image_size, image_size),
    batch_size=batch_size,
    class_mode='categorical'
)

#validation_path = 'cifar-10-data-set-grey-scale/test'
validation_path = 'cifar-10-data-set/test'

val_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    validation_path,
#     validation_split=0.2,
#     subset="validation",
    seed=1337,
    image_size=image_size,
    batch_size=batch_size,
)

# Construindo o modelo

In [ ]:
# TRANSFER LEARNING

model = Sequential()

#model = model.build(input_shape=(32,32))

# Incluindo camada de convolução
model.add(tf.keras.layers.Conv2D(
    filters=12,
    kernel_size=16,
    #strides=(1, 1),
    padding="valid",
    data_format=None,
    dilation_rate=(1, 1),
    groups=1,
    activation='relu',
    use_bias=True,
    kernel_initializer="glorot_uniform",
    bias_initializer="zeros",
    kernel_regularizer=None,
    bias_regularizer=None,
    activity_regularizer=None,
    kernel_constraint=None,
    bias_constraint=None,
))

model.add(tf.keras.layers.MaxPooling2D(pool_size=(16, 16), strides=None, padding="valid", data_format=None))

#Definindo a 2ª camada (Utilizando a aplicação ResNet50)
model.add(ResNet50(include_top = False, pooling = 'avg', weights = 'imagenet'))

# Definindo a 2ª camada como Densa para classificação de 7 classes usando a ativação do SoftMax
model.add(Dense(NUM_CLASSES, activation = 'softmax'))

In [ ]:
model.summary()

In [ ]:
#Compilando o modelo de transferência de aprendizagem (Otimizador SGD)
sgd = optimizers.SGD(lr = 0.01, decay = 1e-6, momentum = 0.9, nesterov = True)
model.compile(optimizer = sgd, loss = OBJECTIVE_FUNCTION, metrics = LOSS_METRICS)

In [ ]:

# Número máximo de etapas que esses geradores terão oportunidade de processar seu conteúdo de origem
# len (train_generator) deve ser 'nº de imagens de trem disponíveis / BATCH_SIZE_TRAINING '
# len (valid_generator) deve ser 'nº de imagens de trem disponíveis / BATCH_SIZE_VALIDATION '

(len(train_dataset), len(val_dataset))
#(BATCH_SIZE_TRAINING, len(train_generator), BATCH_SIZE_VALIDATION)

In [ ]:
# EarlyStopping => Pare o treinamento quando uma métrica monitorada parar de melhorar.
# Assumindo que o objetivo de um treinamento é minimizar a perda. Com isso, a métrica a ser monitorada seria 'loss',
#   e o modo seria 'min'. Um model.fit()loop de treinamento verificará ao final de cada época se a perda não está mais 
#   diminuindo, considerando o min_deltae patiencese aplicável. Uma vez que não esteja mais diminuindo, model.stop_trainingé
#   marcado como Verdadeiro e o treinamento termina.

cb_early_stopper = EarlyStopping(monitor = 'val_loss', patience = EARLY_STOP_PATIENCE)

# ModelCheckpoint => Retorno de chamada para salvar o modelo Keras ou pesos do modelo com alguma frequência.
# ModelCheckpointO retorno de chamada é usado em conjunto com o treinamento model.fit()
#  para salvar um modelo ou pesos (em um arquivo de ponto de verificação) em algum intervalo, 
#  para que o modelo ou pesos possam ser carregados posteriormente para continuar o treinamento do estado salvo.

cb_checkpointer = ModelCheckpoint(filepath = 'pesos/rede.hdf5', monitor = 'val_loss', save_best_only = True, mode = 'auto')

In [ ]:
fit_history = model.fit(
    x = train_dataset,
    steps_per_epoch=STEPS_PER_EPOCH_TRAINING,
    epochs = NUM_EPOCHS,
    validation_data=val_dataset,
    validation_steps=STEPS_PER_EPOCH_VALIDATION,
    callbacks=[cb_checkpointer, cb_early_stopper]
)

# LOAD_WEIGHTS => O savefile inclui:
# A arquitetura do modelo, permitindo reinstanciar o modelo.
# Os pesos do modelo.
# O estado do otimizador, permitindo retomar o treinamento exatamente de onde você parou.
# Isso permite que você salve todo o estado de um modelo em um único arquivo.

In [ ]:
# Métricas de treinamento
print(fit_history.history.keys())

In [ ]:
plt.figure(1, figsize = (15,8)) 
    
plt.subplot(221)  
plt.plot(fit_history.history['accuracy'])  
plt.plot(fit_history.history['val_accuracy'])  
plt.title('model accuracy')  
plt.ylabel('accuracy')  
plt.xlabel('epoch')  
plt.legend(['train', 'valid'])
    
plt.subplot(222)  
plt.plot(fit_history.history['loss'])  
plt.plot(fit_history.history['val_loss'])  
plt.title('model loss')  
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'valid'])

plt.show()